# Final evaluation & Grad-CAM — Oxford-IIIT Pet (Colab GPU)

Runs Member 3's evaluation pipeline on the **best model (frozen MobileNetV2)** and produces the figures the report needs:

- test-set **accuracy / macro F1 / top-3**
- **confusion matrix** (row-normalized)
- **most-confused breed pairs**
- **Grad-CAM** overlays + a correct/incorrect prediction gallery

**Step 0 — turn on the GPU:** menu **Runtime → Change runtime type → GPU (T4)** → Save. Then run the cells top to bottom (~8–12 min, includes an ~800 MB dataset download).

> The checkpoint is trained fresh in this notebook because `.pth` files are not stored in the repo (too large). Frozen MobileNetV2 converges in a few minutes on a GPU.

In [ ]:
# 1) Confirm the GPU is on.
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2) Clone the repo and install. --no-deps keeps Colab's CUDA torch;
#    grad-cam (for Grad-CAM) is installed separately and does not touch torch.
!git clone https://github.com/jeqia0005-debug/oxford-pet-breed-classifier.git
%cd oxford-pet-breed-classifier
!pip install -q -e . --no-deps
!pip install -q grad-cam

In [ ]:
# 3) Train the frozen MobileNetV2 (best model). Downloads the data on the
#    first run and saves checkpoints/mobilenet_frozen.pth.
!python scripts/train_mobilenet_frozen.py --config configs/mobilenet_frozen.yaml

In [ ]:
# 4) Evaluate it on the held-out TEST set. Saves:
#    reports/evaluation_mobilenet_frozen.json
#    reports/figures/confusion_mobilenet_frozen.png
#    reports/figures/gallery_mobilenet_frozen.png
#    reports/figures/gradcam_mobilenet_frozen.png
!python scripts/evaluate_model.py \
    --checkpoint checkpoints/mobilenet_frozen.pth \
    --architecture mobilenet_v2 \
    --name mobilenet_frozen \
    --gradcam-samples 8

In [ ]:
# 5) Show the results and figures.
import json
from IPython.display import Image, display

with open('reports/evaluation_mobilenet_frozen.json') as f:
    res = json.load(f)
print('Top-1 accuracy :', round(res['top1_accuracy'], 4))
print('Top-3 accuracy :', round(res['top3_accuracy'], 4))
print('Macro F1       :', round(res['macro_f1'], 4))
print('\nMost-confused breed pairs:')
for p in res['most_confused_pairs'][:8]:
    print(f"  {p['true_breed']} -> {p['predicted_breed']} ({p['count']})")

for name in ['confusion', 'gradcam', 'gallery']:
    display(Image(f'reports/figures/{name}_mobilenet_frozen.png'))

In [ ]:
# 6) (optional) Download the results + figures for the report.
from google.colab import files
files.download('reports/evaluation_mobilenet_frozen.json')
for name in ['confusion', 'gradcam', 'gallery']:
    files.download(f'reports/figures/{name}_mobilenet_frozen.png')

**Last step:** the metrics (`reports/evaluation_mobilenet_frozen.json`) and figures (`reports/figures/`) are saved and downloaded in the previous cell. Add them to the report's evaluation section.

*Custom CNN test metrics are already in `reports/custom_cnn_results.json` (produced by `train_custom_cnn.py`), so you don't need to re-evaluate it here.*